In [2]:
tabla='ctpco10'
col_essi='fec_pro'
col_tabla='proconfec'
col_dat='fec_pro'
dat='dat_cext004_essi'
essi='essi_dat_cex004_etl'


In [3]:
from decouple import config
import decouple #eliminar en .py
from sqlalchemy import create_engine
import pandas as pd
from datetime import datetime, timedelta
import time 
from datetime import datetime
from sqlalchemy import text
import oracledb
import sys

In [4]:
inicioTiempo = time.time()
inicioProceso=time.time()
now_inicio = datetime.now()

In [5]:
######################FUNCIONES DE LOG###########################
global dim, control_a, control_d, base1, falla, merge
control_a=[]
control_d=[]
dim=[]
falla=[]
id_afectado=[]

def log_falla(id, cod, isNeeded):
    if isNeeded:
        filas_perdidas = merge.loc[pd.isnull(merge[id])]
        filas_perdidas = filas_perdidas.drop_duplicates(subset=[cod])
        filas_perdidas=filas_perdidas[[cod]]
        if filas_perdidas.empty:
            filas_perdidas_string = 0
        else:
            filas_perdidas_string = filas_perdidas.to_string(index=False, header=False)
            filas_perdidas_string = filas_perdidas_string.replace('\n', ',')
    else:
        filas_perdidas_string = 0
    falla.append(filas_perdidas_string)
    id_afectado.append(id)

def log_control(table):    
    dim.append(table)
    control_d.append(base1.shape[0])
    control_a.append(base1.shape[0])

In [6]:
config = decouple.AutoConfig(' ')
DB_USER=config("USER2_BDI_POSTGRES")
DB_PASSWORD=config("PASS2_BDI_POSTGRES")
DB_NAME="essi_etl"
DB_PORT="5432"
DB_HOST=config("HOST2_BDI_POSTGRES")
cadena1  = f"postgresql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
engine1 = create_engine(cadena1)
connection1 = engine1.connect()

DB_USER=config("USER2_BDI_POSTGRES")
DB_PASSWORD=config("PASS2_BDI_POSTGRES")
DB_NAME="dw_essalud"
DB_PORT="5432"
DB_HOST=config("HOST2_BDI_POSTGRES")
cadena2  = f"postgresql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
engine2 = create_engine(cadena2)
connection2 = engine2.connect()

DB_USER=config("USER2_BDI_POSTGRES")
DB_PASSWORD=config("PASS2_BDI_POSTGRES")
DB_NAME="etl_logs"
DB_PORT="5432"
DB_HOST=config("HOST2_BDI_POSTGRES")
cadena3  = f"postgresql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
engine3 = create_engine(cadena3)
connection3 = engine3.connect()

In [7]:
from datetime import datetime
from sqlalchemy import text


fecha = pd.read_sql_query("SELECT fec_ini FROM etl_act where id_mod=6", con=connection2)
fecha= fecha.iloc[0, 0]
print(fecha)

now = datetime.now()

query=f"UPDATE etl_act SET fec_act ='{now}' WHERE id_mod=6"

c1= text(query)
connection2.execute(c1)

2023-06-01


In [8]:
base1=pd.read_sql_query(f"SELECT * FROM {essi} WHERE {col_essi} >='{fecha}'", con=connection1)

In [9]:
base1.columns

Index(['ori_cas', 'cod_cas', 'des_cas', 'cod_red', 'des_red', 'cod_ser',
       'des_ser', 'cod_act', 'des_act', 'cod_sub', 'des_sub', 'tip_doc',
       'num_doc', 'fec_pro', 'cod_con', 'cup_vol', 'cup_rec', 'cup_int',
       'cup_dia', 'top_nue', 'top_adi', 'top_ref', 'top_ter', 'oto_vol',
       'oto_rec', 'oto_int', 'oto_top_nue', 'oto_top_adi', 'oto_top_ref',
       'oto_top_ter', 'cit_eli', 'cit_ate', 'cit_rep', 'dem_ins', 'est_com',
       'reg_cod', 'cup_ref', 'oto_ref', 'pac_hor', 'cod_are', 'des_are',
       'hor_ini', 'hor_fin'],
      dtype='object')

In [10]:
base1.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 778164 entries, 0 to 778163
Data columns (total 43 columns):
 #   Column       Non-Null Count   Dtype  
---  ------       --------------   -----  
 0   ori_cas      778164 non-null  object 
 1   cod_cas      778164 non-null  object 
 2   des_cas      778164 non-null  object 
 3   cod_red      778164 non-null  object 
 4   des_red      778164 non-null  object 
 5   cod_ser      778164 non-null  object 
 6   des_ser      778164 non-null  object 
 7   cod_act      778164 non-null  object 
 8   des_act      778164 non-null  object 
 9   cod_sub      778164 non-null  object 
 10  des_sub      778164 non-null  object 
 11  tip_doc      778164 non-null  object 
 12  num_doc      778164 non-null  object 
 13  fec_pro      778164 non-null  object 
 14  cod_con      778164 non-null  object 
 15  cup_vol      778164 non-null  float64
 16  cup_rec      778164 non-null  float64
 17  cup_int      778164 non-null  float64
 18  cup_dia      778164 non-

In [11]:
base1.shape

(778164, 43)

In [12]:
base2=pd.read_sql_query(f"SELECT * FROM {dat} LIMIT 10", con=connection2)

In [13]:
tiempo = pd.read_sql_query(f"SELECT id_tiempo,dt_fecha,dt_mes,dt_dia,dt_dia_sem,dt_sem,dt_ano FROM dim_tiempo", con=connection2)
tiempo=tiempo.rename(columns={"id_tiempo":"id_time_pro","dt_fecha":"fec_pro","dt_mes":"mes_pro","dt_dia":"dia_pro","dt_dia_sem":"dia_sem_pro","dt_sem":"sem_pro","dt_ano":"ano_pro"})
base1['fec_pro'] = pd.to_datetime(base1['fec_pro']).dt.date

base1=pd.merge(base1,tiempo,how='left',on='fec_pro')


In [14]:
control_a.append(base1.shape[0])

In [15]:
are = pd.read_sql_query(f"SELECT id_area,cod_are FROM dim_areas", con=connection2)
merge=pd.merge(base1,are,how='left',on='cod_are')
base1=pd.merge(base1,are,how='inner',on='cod_are')
base1.shape

(778164, 50)

In [16]:
log_falla('id_area', 'cod_are', True)
log_control('dim_areas')
base1=base1.drop('cod_are',axis=1)

In [17]:
red = pd.read_sql_query(f"SELECT id_red,cod_red FROM dim_red", con=connection2)
merge=pd.merge(base1,red,how='left',on='cod_red')
base1=pd.merge(base1,red,how='inner',on='cod_red')
base1.shape

(778164, 50)

In [18]:
log_falla('id_red', 'cod_red', True)
log_control('dim_red')
base1=base1.drop('cod_red',axis=1)

In [19]:
consult = pd.read_sql_query(f"SELECT id_consult,cod_cas,cod_con FROM dim_consult", con=connection2)
consult["KEY"]=consult["cod_con"].str.strip()+consult["cod_cas"].str.strip()
consult=consult.drop(["cod_con",'cod_cas'], axis=1)
base1["KEY"]=base1["cod_con"].astype(str).str.strip()+base1['cod_cas'].astype(str).str.strip()
merge = pd.merge(base1,consult,how='left',on='KEY')
base1 = pd.merge(base1,consult,how='inner',on='KEY')
base1=base1.rename(columns={"id_consult":"id_consul"})
base1.shape

(777398, 51)

In [20]:
log_falla('id_consult', 'KEY', True)
log_control('dim_consult')
base1=base1.drop('KEY', axis=1)

In [21]:
cas = pd.read_sql_query(f"SELECT id_cas,cod_cas FROM dim_cas where id_red is not null", con=connection2)
merge=pd.merge(base1,cas,how='left',on='cod_cas')
base1=pd.merge(base1,cas,how='inner',on='cod_cas')
base1.shape

(777398, 51)

In [22]:
log_falla('id_cas', 'cod_cas', True)
log_control('dim_cas')
base1=base1.drop('cod_cas',axis=1)

In [23]:
serv= pd.read_sql_query(f"SELECT id_serv,cod_ser FROM dim_servicios", con=connection2)
merge=pd.merge(base1,serv,how='left',on='cod_ser')
base1=pd.merge(base1,serv,how='inner',on='cod_ser')
base1.shape

(777398, 51)

In [24]:
log_falla('id_serv', 'cod_ser', True)
log_control('dim_servicios')
base1=base1.drop('cod_ser',axis=1)

In [25]:
oricas = pd.read_sql_query(f"SELECT id_oricas,ori_cod FROM dim_oricas", con=connection2)
oricas=oricas.rename(columns={"ori_cod":"ori_cas"})
oricas=oricas.rename(columns={"id_oricas":"id_ori"})
merge=pd.merge(base1,oricas,how='left',on='ori_cas')
base1=pd.merge(base1,oricas,how='inner',on='ori_cas')
base1.shape

(777398, 51)

In [26]:
log_falla('id_ori', 'ori_cas', True)
log_control('dim_oricas')
base1=base1.drop('ori_cas',axis=1)

In [27]:
tipdoc = pd.read_sql_query(f"SELECT id_tipdoc,cod_tdo FROM dim_tipdoc", con=connection2)
tipdoc=tipdoc.rename(columns={"cod_tdo":"tip_doc"})
merge=pd.merge(base1,tipdoc,how='left',on='tip_doc')
base1=pd.merge(base1,tipdoc,how='inner',on='tip_doc')
base1.shape

(777398, 51)

In [28]:
log_falla('id_tipdoc', 'tip_doc', True)
log_control('dim_tipdoc')
base1=base1.drop('tip_doc',axis=1)

In [29]:
numdoc = pd.read_sql_query(f"SELECT id_person,num_doc FROM dim_personal", con=connection2)
numdoc=numdoc.drop_duplicates(subset="num_doc")
numdoc=numdoc.rename(columns={"id_person":"id_persona"})
merge=pd.merge(base1,numdoc,how='left',on='num_doc')
base1=pd.merge(base1,numdoc,how='inner',on='num_doc')
base1.shape

(777398, 51)

In [30]:
log_falla('id_persona', 'num_doc', True)
log_control('dim_personal')
base1=base1.drop('num_doc',axis=1)

In [31]:
base1.columns

Index(['des_cas', 'des_red', 'des_ser', 'cod_act', 'des_act', 'cod_sub',
       'des_sub', 'fec_pro', 'cod_con', 'cup_vol', 'cup_rec', 'cup_int',
       'cup_dia', 'top_nue', 'top_adi', 'top_ref', 'top_ter', 'oto_vol',
       'oto_rec', 'oto_int', 'oto_top_nue', 'oto_top_adi', 'oto_top_ref',
       'oto_top_ter', 'cit_eli', 'cit_ate', 'cit_rep', 'dem_ins', 'est_com',
       'reg_cod', 'cup_ref', 'oto_ref', 'pac_hor', 'des_are', 'hor_ini',
       'hor_fin', 'id_time_pro', 'mes_pro', 'dia_pro', 'dia_sem_pro',
       'sem_pro', 'ano_pro', 'id_area', 'id_red', 'id_consul', 'id_cas',
       'id_serv', 'id_ori', 'id_tipdoc', 'id_persona'],
      dtype='object')

In [32]:

subacti = pd.read_sql_query(f"SELECT id_subacti,cod_sub,cod_act FROM dim_subacti", con=connection2)
subacti["KEY"]=subacti["cod_sub"]+subacti["cod_act"]
subacti=subacti.drop(["cod_sub",'cod_act'], axis=1)
base1["KEY"]=base1["cod_sub"].astype(str)+base1['cod_act'].astype(str)
base1["KEY"]=base1["KEY"].str.replace(' ', '', regex=True)
subacti["KEY"]=subacti["KEY"].str.replace(' ', '', regex=True)
merge = pd.merge(base1,subacti,how='left',on='KEY')
base1 = pd.merge(base1,subacti,how='inner',on='KEY')
base1.shape

(777398, 52)

In [33]:
log_falla('id_subacti', 'KEY', True)
log_control('dim_subacti')
base1=base1.drop('KEY', axis=1)

In [34]:
activi = pd.read_sql_query(f"SELECT id_activi,cod_act FROM dim_activi", con=connection2)
activi=activi.rename(columns={"id_activi":"id_acti"})
merge=pd.merge(base1,activi,how='left',on='cod_act')
base1=pd.merge(base1,activi,how='inner',on='cod_act')
base1.shape

(777398, 52)

In [35]:
log_falla('id_acti', 'cod_act', True)
log_control('dim_activi')
base1=base1.drop('cod_act', axis=1)

In [36]:
base1.columns

Index(['des_cas', 'des_red', 'des_ser', 'des_act', 'cod_sub', 'des_sub',
       'fec_pro', 'cod_con', 'cup_vol', 'cup_rec', 'cup_int', 'cup_dia',
       'top_nue', 'top_adi', 'top_ref', 'top_ter', 'oto_vol', 'oto_rec',
       'oto_int', 'oto_top_nue', 'oto_top_adi', 'oto_top_ref', 'oto_top_ter',
       'cit_eli', 'cit_ate', 'cit_rep', 'dem_ins', 'est_com', 'reg_cod',
       'cup_ref', 'oto_ref', 'pac_hor', 'des_are', 'hor_ini', 'hor_fin',
       'id_time_pro', 'mes_pro', 'dia_pro', 'dia_sem_pro', 'sem_pro',
       'ano_pro', 'id_area', 'id_red', 'id_consul', 'id_cas', 'id_serv',
       'id_ori', 'id_tipdoc', 'id_persona', 'id_subacti', 'id_acti'],
      dtype='object')

In [37]:
base1['reg_cod']

0         1  
1         1  
2         1  
3         1  
4         1  
         ... 
777393    1  
777394    1  
777395    1  
777396    1  
777397    1  
Name: reg_cod, Length: 777398, dtype: object

In [38]:
estreg = pd.read_sql_query(f"SELECT id_estreg,cod_reg FROM dim_cexestreg", con=connection2)
estreg=estreg.rename(columns={"cod_reg":"reg_cod"})
estreg['reg_cod'] = estreg['reg_cod'].str.strip()
base1['reg_cod']=base1['reg_cod'].str.strip()
merge=pd.merge(base1,estreg,how='left',on='reg_cod')
base1=pd.merge(base1,estreg,how='inner',on='reg_cod')
base1.shape

(777398, 52)

In [39]:
log_falla('id_estreg', 'reg_cod', True)
log_control('dim_cexestreg')
base1=base1.drop('reg_cod', axis=1)

In [40]:
id_atecup = pd.read_sql_query(f"SELECT id_atecup,est_com FROM dim_cexatecup", con=connection2)
merge=pd.merge(base1,id_atecup,how='left',on='est_com')
base1=pd.merge(base1,id_atecup,how='inner',on='est_com')
base1.shape

(777398, 52)

In [41]:
log_falla('id_atecup', 'est_com', True)
base1=base1.drop('est_com',axis=1)
dim.append('dim_cexatecup')
control_d.append(base1.shape[0])

In [42]:
base1['horfin']=base1['hor_fin'].astype(str).str[10:16]
base1['horini']=base1['hor_ini'].astype(str).str[10:16]
base1['horfin'] = base1['horfin'].str.strip()
base1['horfin'] = pd.to_datetime(base1['horfin'], format='%H:%M')
base1['horini'] = base1['horini'].str.strip()
base1['horini'] = pd.to_datetime(base1['horini'], format='%H:%M')
base1['diferencia_horas'] = base1['horfin'] - base1['horini']
base1['horas']=base1['diferencia_horas'].astype(str).str[-9:-6]

In [43]:
base1.columns

Index(['des_cas', 'des_red', 'des_ser', 'des_act', 'cod_sub', 'des_sub',
       'fec_pro', 'cod_con', 'cup_vol', 'cup_rec', 'cup_int', 'cup_dia',
       'top_nue', 'top_adi', 'top_ref', 'top_ter', 'oto_vol', 'oto_rec',
       'oto_int', 'oto_top_nue', 'oto_top_adi', 'oto_top_ref', 'oto_top_ter',
       'cit_eli', 'cit_ate', 'cit_rep', 'dem_ins', 'cup_ref', 'oto_ref',
       'pac_hor', 'des_are', 'hor_ini', 'hor_fin', 'id_time_pro', 'mes_pro',
       'dia_pro', 'dia_sem_pro', 'sem_pro', 'ano_pro', 'id_area', 'id_red',
       'id_consul', 'id_cas', 'id_serv', 'id_ori', 'id_tipdoc', 'id_persona',
       'id_subacti', 'id_acti', 'id_estreg', 'id_atecup', 'horfin', 'horini',
       'diferencia_horas', 'horas'],
      dtype='object')

In [44]:
df1_columns = set(base1.columns)
df2_columns = set(base2.columns) 
different_columns = df2_columns - df1_columns
different_columns

set()

In [45]:
borrando=f"DELETE FROM {dat} WHERE {col_dat} >='{fecha}'"
borrado = connection2.execute(borrando)

In [46]:
query_count_before = f"SELECT COUNT(*) FROM {dat}"
cant_antes = connection2.execute(query_count_before).fetchone()[0]
print(f"Cantidad de filas en la tabla {dat} antes de la inserción: {cant_antes}")

Cantidad de filas en la tabla dat_cext004_essi antes de la inserción: 447275


In [47]:
comunes = set(base1.columns).intersection(set(base2.columns)) 
base = base1[list(comunes)]
base.to_sql(name=f'{dat}', con=engine2, if_exists='append', index=False,chunksize=5000)

155398

In [48]:

query_count_after = f"SELECT COUNT(*) FROM {dat}"
cant_despues = connection2.execute(query_count_after).fetchone()[0]
print(f"Cantidad de filas en la tabla {dat} después de la inserción: {cant_despues}")
print(f"La cantidad de filas insertadas fue de: {cant_despues-cant_antes}")


Cantidad de filas en la tabla dat_cext004_essi después de la inserción: 1224673
La cantidad de filas insertadas fue de: 777398


In [49]:
#base=base.sort_values("fec_pro",ascending=False)
#fecha_fin= base.iloc[6, 1]
fecha_fin = "2024-12-31"

In [50]:
proceso = pd.read_sql_query("SELECT des_mod FROM etl_act where id_mod=6", con=connection2)
proceso = proceso.iloc[0, 0]
cod_proceso = pd.read_sql_query("SELECT id_mod FROM etl_act where id_mod=6", con=connection2)
cod_proceso = cod_proceso.iloc[0, 0]

now_fin = datetime.now()
fecha_log = now.strftime("%Y-%m-%d")
hora_log_inicio = now_inicio.strftime("%H:%M")
hora_log_fin = now_fin.strftime("%H:%M")
tabla_logs = pd.DataFrame({'esperado':control_a,'obtenido':control_d,'dim':dim,'falla':falla})
tabla_logs['proceso']=proceso
tabla_logs['dat']=dat
tabla_logs['fecha_ejecucion']=fecha_log
tabla_logs['hora_inicio']=hora_log_inicio
tabla_logs['hora_fin']=hora_log_fin
tabla_logs['faltante']=tabla_logs['esperado']-tabla_logs['obtenido']
tabla_logs['codigo']=cod_proceso
tabla_logs['fecha_ini']=fecha
tabla_logs['fecha_ter']=fecha_fin
tabla_logs['id_afectado']=id_afectado

nuevas_columnas = ['codigo', 'proceso', 'dat', 'fecha_ejecucion', 'hora_inicio','hora_fin', 'dim', 'fecha_ini','fecha_ter','esperado', 'obtenido', 'faltante','falla', 'id_afectado']

tabla_logs = tabla_logs.reindex(columns=nuevas_columnas)

In [51]:
tabla_logs

,codigo,proceso,dat,fecha_ejecucion,hora_inicio,hora_fin,dim,fecha_ini,fecha_ter,esperado,obtenido,faltante,falla,id_afectado
0,6.0,CONSULTA EXTERNA CUPOS ...,dat_cext004_essi,2023-06-02,10:19,10:25,dim_areas,2023-06-01,2024-12-31,778164,778164,0,0,id_area
1,6.0,CONSULTA EXTERNA CUPOS ...,dat_cext004_essi,2023-06-02,10:19,10:25,dim_red,2023-06-01,2024-12-31,778164,778164,0,0,id_red
2,6.0,CONSULTA EXTERNA CUPOS ...,dat_cext004_essi,2023-06-02,10:19,10:25,dim_consult,2023-06-01,2024-12-31,778164,777398,766,"C218.016,C-309408, LAB039,TM-15205,TM-16205, ...",id_consult
3,6.0,CONSULTA EXTERNA CUPOS ...,dat_cext004_essi,2023-06-02,10:19,10:25,dim_cas,2023-06-01,2024-12-31,777398,777398,0,0,id_cas
4,6.0,CONSULTA EXTERNA CUPOS ...,dat_cext004_essi,2023-06-02,10:19,10:25,dim_servicios,2023-06-01,2024-12-31,777398,777398,0,0,id_serv
5,6.0,CONSULTA EXTERNA CUPOS ...,dat_cext004_essi,2023-06-02,10:19,10:25,dim_oricas,2023-06-01,2024-12-31,777398,777398,0,0,id_ori
6,6.0,CONSULTA EXTERNA CUPOS ...,dat_cext004_essi,2023-06-02,10:19,10:25,dim_tipdoc,2023-06-01,2024-12-31,777398,777398,0,0,id_tipdoc
7,6.0,CONSULTA EXTERNA CUPOS ...,dat_cext004_essi,2023-06-02,10:19,10:25,dim_personal,2023-06-01,2024-12-31,777398,777398,0,0,id_persona
8,6.0,CONSULTA EXTERNA CUPOS ...,dat_cext004_essi,2023-06-02,10:19,10:25,dim_subacti,2023-06-01,2024-12-31,777398,777398,0,0,id_subacti
9,6.0,CONSULTA EXTERNA CUPOS ...,dat_cext004_essi,2023-06-02,10:19,10:25,dim_activi,2023-06-01,2024-12-31,777398,777398,0,0,id_acti


In [52]:
tabla_logs.to_sql(name=f'logs', con=connection3, if_exists='append', index=False,chunksize=10000)

12

In [53]:
connection1.close()
connection2.close()
connection3.close()


In [54]:
#base.shape